In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [ ]:
boston = fetch_openml(name="boston", version=1, as_frame=True)
X = boston.data
y = boston.target

In [ ]:
# Normalizing data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Splitting on train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [ ]:
# Definir red neuronal profunda
class DeepRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(DeepRegressionModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# Crear el modelo
input_dim = X_train.shape[1]
model = DeepRegressionModel(input_dim)

In [ ]:
# Definir pérdida y optimizador
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Entrenar modelo
n_epochs = 800
losses = []

for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()

    # Forward
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    # Backward
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{n_epochs}] - Loss: {loss.item():.4f}")


In [ ]:
# Evaluación
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor)
    mse = mean_squared_error(y_test_tensor, y_pred)
    r2 = r2_score(y_test_tensor, y_pred)
    print(f"\n🔹 MSE en test: {mse:.4f}")
    print(f"🔹 R² en test: {r2:.4f}")

In [ ]:
# Gráficas
plt.figure(figsize=(12, 5))

# Curva de pérdida
plt.subplot(1, 2, 1)
plt.plot(losses, label="Training Loss", color="blue")
plt.xlabel("Épocas")
plt.ylabel("MSE Loss")
plt.title("Curva de pérdida durante el entrenamiento")
plt.legend()

# Real vs Predicho
plt.subplot(1, 2, 2)
plt.scatter(y_test_tensor, y_pred, alpha=0.7, color="orange")
plt.xlabel("Valores reales")
plt.ylabel("Predicciones")
plt.title("Real vs Predicho (Boston Housing)")
plt.plot(
    [y_test_tensor.min(), y_test_tensor.max()],
    [y_test_tensor.min(), y_test_tensor.max()],
    'r--'
)

plt.tight_layout()
plt.show()
